# LangGraph의 주요 컴포넌트
## 스테이트: 그래프의 상태표현
- 스테이트란 LangGraph 워크플로에서 실행되는 각 노드에 의해 **업데이트된 값을 저장**하기 위한 매키니즘
- 각 노드는 이 스테이트에 저장된 데이터를 읽고 쓰면서 처리를 진행함
- 따라서 LangGraph로 구현을 할때는 우선 어떤 데이터구조로 데이터를 저장할 필요가 있는지 설계해야함

In [1]:
import operator
from typing import Annotated
from pydantic import BaseModel, Field

class State(BaseModel):
    """
    스테이트의 각 필드에서는 업데이터 시 연산을 어노테이션을 사용해 명시적으로 지정가능
    기본적으로 set 연산이 사용되어 해당하는 값이 덮어쓰기 됨
    """

    query: str = Field(..., description="사용자의 질문")
    current_role: str = Field(default="", description="선정된 답변 역할")
    messages: Annotated[list[str], operator.add] = Field(default=[], description="답변 이력")  # operator.add로 리스트에 요소 추가가 수행됨
    current_judge: bool = Field(default=False, description="품질 검사 결과")
    judgement_reason: str = Field(default="", description="품질 검사 판정 이유")




- 스테이트에 대한 타입 정의가 완료되면, `StateGraph` 클래스에 그 정의를 전달하고 그래프의 인스턴스를 생성
- `StateGraph` 클래스는 LangGraph에서 그래프 구조 정의를 위해 사용되는 클래스로, 워크플로를 구성하는 노드와 에지 역할을 수행

In [2]:
from langgraph.graph import StateGraph

workflow = StateGraph(State)

## 노드: 그래프를 구성하는 처리 단위
- 그래프를 구성하는 노드는 `StateGraph`클래스의 `add_node`함수를 사용해 추가

### 노드의 지정방법


```py
# 함수 또는 Runnable 지정
workflow.add_node(answering_node)

# 노드 이름과 (함수 또는 Runnable) 쌍을 지정
workflow.add_node("answering", answering_node)

```


### 노드의 구현 방법
- 노드에  함수를 전달하는 경우, 스테이트 객체를 인자로 받고 변경분을 나타내는 딕셔너리 타입의 객체를 반환하도록 구현

In [3]:
from typing import Any

def answering_node(state:State) -> dict[str, Any]:
    """
    query(사용자의 질문 내용)와 role(선정된 역할)을 추출하고 이를 기반으로 답변생성
    """
    query: state.query
    role: state.current_role

    generated_message = "...생성 처리"

    return {"messages" : [generated_message]} # messages 필드의 리스트에 추가하기 위해 리스트타입으로 반환

- Runnable을 전달하는 경우에도 함수와 마찬가지로 스테이트를 받아 변경분을 반환하는 구현으로 해야함

    ```py
    prompt =" ..."
    llm = "..."

    answering_node = (
        RunnalbePassThrough.assign(
            query=lambda state: state.query,
            role=lambda state: state.role
        )
        | prompt
        | llm 
        | StrOutputParser()
        | RunnablePassThrough.assign(messages=lambda x: [x])
    )
    ```

- 또한 여러 필드를 업데이트하는 경우에는 여러 필드 이름과 대응하는 키에 값을 설정한 딕셔너리 타입의 객체를 반환

    ```py
    def check_node(state: State)->dict[str, Any]:
        query = state.query
        message = state.messages[-1]

        judge = "...판정결과..."
        reason = "...이유 생성..."

        return {"current_judge" : judge, "judgement_reason": reason}
    ```

## 에지: 노드 간의 연결
### 1. 엔트리 포인트
- 그래프의 시작노드를 지정하는 에지
    ```py
    workflow.set_entry_point("selection")
    workflow.add_edge(START, "selection")
    ```

### 2. 에지
- 특정 노드에서 다른 노드로 무조건 전이하는 에지.
- 첫번째 인자에 전이 시작노드의 이름, 두번째 인자에 전이 대상노드의 이름을 문자열로 지정
    ```py
    workflow.add_edge("selection", "answering")
    ```

### 3. 조건부 에지
- 조건에 기반해 전이할 노드를 결정. `add_conditional_edges` 함수로 설정
- 첫번쨰 인자: 전이 시작 노드를 문자열로 지정
- 두번째 인자: 어떤 값을 반환하는 함수를 설정
- 세번째 인자: 두번째 인자에서 반환되는 값에 대응하는 전이 대상 노드이 이름과의 매핑을 딕셔너리 타입의 객체로 설정
    ```py
    from langgraph.graph import END
    workflow.add_conditional_edges(
        "check",
        lambda state: state.current_judge,
        {True: END, False: "selection"} # state.current_judge의 값이 True면 종점으로, False면 "selection" 노드로 전이
    )
    ```

## 컴파일된 그래프
- 정의된 그래프는 `compile` 함수를 통해 실행 가능한 `CompiledGraph` 인스턴스로 변환

    ```py
    compiled = workflow.compile()
    ```
- `CompiledGraph` 클래스의 인스턴스는 `Runnable`로 실행할 수 있음. 따라서, `invoke`, `stream` 등의 함수를 사용해 정의된 그래프를 실행할 수 있음

    ```py
    initial_state = State(query="생성 AI에 대해 알려주세요")
    result = compiled.invoke(initial_state)

    result = await compiled.ainvoke(initial_state) # 비동기 함수

    for step in compiled.stream(initial_state):
        print(step)
    ```



# 실습: Q&A 애플리케이션
## OpenAI API 키 설정

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

LANGSMITH_TRACING=os.getenv("LANGSMITH_TRACING")
LANGSMITH_ENDPOINT=os.getenv("LANGSMITH_ENDPOINT")
LANGSMITH_API_KEY=os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT=os.getenv("LANGSMITH_PROJECT")

## 역할 정의    

In [5]:
ROLES = {
    "1":{
        "name":"일반 지식 전문가",
        "description":"폭넓은 분야의 일반적인 질문에 답변",
        "details":"폭넓은 분야의 일반적인 질문에 대해 정확하고 이해하기 쉬운 답변을 제공하세요"
    },
    "2" : {
        "name": "생성형 AI 제품 전문가",
        "description": "생성형 AI와 관련 제품, 기술에 관한 전문적인 질문에 답변",
        "details" : "생성형 AI와 관련 제품, 기술에 관한 전문적인 질문에 대해 최신 정보와 깊은 통찰력을 제공하세요"
    },
    "3":{
        "name": "카운슬러",
        "description":"개인적인 고민이나 심리적인 문제에 대해 지원 제공",
        "details":"개인적인 고민이나 심리적인 문제에 대해 공감적이고 지원적인 답변을 제공하고, 가능하다면 적절한 조언도 해주세요"
    }
}

## 스테이트 정의


In [6]:
import operator
from typing import Annotated
from pydantic import BaseModel, Field

class State(BaseModel):
    """
    스테이트의 각 필드에서는 업데이터 시 연산을 어노테이션을 사용해 명시적으로 지정가능
    기본적으로 set 연산이 사용되어 해당하는 값이 덮어쓰기 됨
    """

    query: str = Field(..., description="사용자의 질문")
    current_role: str = Field(default="", description="선정된 답변 역할")
    messages: Annotated[list[str], operator.add] = Field(default=[], description="답변 이력")  # operator.add로 리스트에 요소 추가가 수행됨
    current_judge: bool = Field(default=False, description="품질 검사 결과")
    judgement_reason: str = Field(default="", description="품질 검사 판정 이유")


## Chat Model 초기화

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import ConfigurableField

llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0.0)
llm = llm.configurable_fields(max_tokens=ConfigurableField(id="max_tokens"))

### selection 노드 구현
- 답변 역할 선정을 수행하는 selection 노드에서는 각 역할에 해당하는 번호만 LLM에 응답하도록 프롬프트 설정
- LLM에 의한 처리 후, 번호에 해당하는 역할 이름을 스테이트에 설정

In [17]:
from typing import Any
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def selection_node(state: State) -> dict[str, Any]:
    query = state.query

    role_options = "\n".join([f"{k}. {v['name']} : {v['description']}" for k, v in ROLES.items()])

    prompt = ChatPromptTemplate.from_template("""
                                              질문을 분석하고, 가장 적절한 답변 담당 역할을 선택하세요.
                                              선택지:
                                              {role_options}

                                              답변은 선택지의 번호(1,2, 또는 3)만 반환하세요.
                                              질문: {query}
    """.strip())

    chain = prompt | llm.with_config(configurable=dict(max_tokens=5)) | StrOutputParser()
    role_number = chain.invoke({"role_options" : role_options, "query" : query})

    selected_role = ROLES[role_number.strip()]["name"]

    return {"current_role" : selected_role}


### answering 노드 구현
- 선정된 역할에 기반해 답변을 수행하는 answering 노드에서는 역할에 기반한 답변을 제공하도록 프롬프트 설정
- LLM이 처리한 후, 응답을 messages 리스트에 추가

In [18]:
def answering_node(state: State) -> dict[str, Any]:
    query = state.query
    role = state.current_role

    role_details = "\n".join([f"- {v['name']}: {v['details']}" for v in ROLES.values()])


    prompt = ChatPromptTemplate.from_template("""
    당신은 {role}로서 답변하세요. 다음 질문에 대해 당신의 역할에 기반한 적절한 답변을 제공하세요.
    역할 상세: {role_details}
    질문: {query}
    답변:

    """.strip())

    chain = prompt | llm | StrOutputParser()

    answer = chain.invoke({"role": role, "role_details" : role_details, "query" : query})
    return {"messages" : [answer]}


### check 노드 구현
- 답변의 품질을 체크하는 check 노드에서는 사용자의 질문과 답변 내용을 바탕으로 품질을 체크하는 프롬프트 설정
- `with_structured_output`을 지정해 생성결과의 내용이 Judgement모델의 내용으로 반환되도록 지시

In [19]:
class Judgement(BaseModel):
    judge: bool = Field(default=False, description="판정 결과")
    reason : str = Field(default="", description="판정 이유")

def check_node(state: State) -> dict[str, Any]:
    query = state.query
    answer = state.messages[-1]

    prompt = ChatPromptTemplate.from_template("""
    다음 답변의 품질을 체크하고, 문제가 있으면 'False', 문제가 없으면 'True'로 답변하세요. 
    또한, 그 판정 이유도 설명하세요.
    사용자의 질문: {query}
    답변: {answer}
                                              
    """.strip())

    chain =  prompt | llm.with_structured_output(Judgement)

    result : Judgement = chain.invoke({"query": query , "answer": answer})

    return {
        "current_judge" : result.judge,
        "judgement_reason" : result.reason
    }

## 그래프 생성

In [20]:
from langgraph.graph import StateGraph
workflow = StateGraph(State)

## 노드 추가

In [21]:
workflow.add_node("selection", selection_node)
workflow.add_node("answering", answering_node)
workflow.add_node("check", check_node)

## 에지 정의

In [22]:
# selection 노드에서 처리 시작
workflow.set_entry_point("selection")

# selection노드에서 answering 노드로
workflow.add_edge("selection", "answering")

# answering 노드에서 check 노드로
workflow.add_edge("answering", "check")

## 조건부 에지 정의


In [23]:
from langgraph.graph import END

# check 노드에서 다음 노드로의 전환에 조건부 에지 정의
# state.current_judge 값이 True면 END 노드로, False면 selection 노드로
workflow.add_conditional_edges(
    "check",
    lambda state : state.current_judge,
    {True: END, False: "selection"}
)

## 그래프 컴파일 

In [24]:
compiled = workflow.compile()

## 그래프 실행

In [25]:
initial_state = State(query="생성형 AI에 관해 알려주세요")
result = compiled.invoke(initial_state)

In [27]:
print(result["messages"][-1])

물론입니다. 생성형 AI(Generative AI)는 **새로운 콘텐츠를 만들어내는 인공지능**입니다. 단순히 정보를 분류하거나 예측하는 데 그치지 않고, **텍스트, 이미지, 음성, 영상, 코드** 같은 결과물을 생성할 수 있습니다.

## 1. 생성형 AI란?
생성형 AI는 대규모 데이터로 학습한 뒤, 그 패턴을 바탕으로 **새로운 내용을 만들어내는 AI**입니다.  
예를 들면:
- 글쓰기: 이메일, 보고서, 기사, 요약문 생성
- 이미지 생성: 설명만 입력하면 그림 생성
- 음성 생성: 자연스러운 TTS(텍스트 음성 변환)
- 영상 생성: 짧은 영상이나 광고 시안 생성
- 코드 생성: 프로그램 코드 작성 및 보조

## 2. 어떻게 작동하나요?
생성형 AI는 보통 다음과 같은 방식으로 작동합니다.
1. **대량의 데이터 학습**: 책, 웹문서, 이미지, 코드 등에서 패턴을 학습
2. **모델이 확률적으로 다음 결과 예측**: 예를 들어 다음 단어, 다음 픽셀, 다음 음절 등을 예측
3. **사용자 요청에 맞게 생성**: 프롬프트(입력 지시문)에 따라 결과를 만듦

대표적인 기술로는:
- **대규모 언어 모델(LLM)**: 텍스트 생성
- **확산모델(Diffusion Model)**: 이미지 생성
- **멀티모달 모델**: 텍스트, 이미지, 음성 등을 함께 이해하고 생성

## 3. 어떤 곳에 쓰이나요?
생성형 AI는 다양한 산업에서 활용됩니다.
- **업무 생산성**: 문서 초안 작성, 회의록 요약, 이메일 작성
- **고객지원**: 챗봇, 상담 자동화
- **마케팅/콘텐츠**: 광고 문구, SNS 콘텐츠, 디자인 시안
- **개발**: 코드 작성, 디버깅, 테스트 보조
- **교육**: 맞춤형 학습, 문제 생성, 설명 보조
- **의료/연구**: 문헌 요약, 데이터 분석 보조, 시뮬레이션 지원

## 4. 장점은 무엇인가요?
- **속도 향상**: 반복 작업을 빠르게 처리
- **창의성 보조**: 아이디어가 필요할 때 다양한 초안 제공
- **비용 